#Clone the Datasets from Github

In [1]:
!git clone https://github.com/DaoPhang/Algo-Project.git
!pip install openpyxl tabulate --quiet

fatal: destination path 'Algo-Project' already exists and is not an empty directory.


# Part 7 — The Phantom Dice

## Step 1 — Mission Problem and Candidate Algorithms

**Resource:** Infiltration sector — one sector the team will physically enter.

**Constraint:** Cypher Nexus has flagged certain sectors as monitored. If the team repeatedly uses a predictable entry point, the enemy can learn the pattern and intercept them.

**Goal:** Select a real movement sector that is safe, deception-aware, and non-deterministic, while also selecting a decoy sector to misdirect surveillance.

### Candidate Algorithms

| Status | Algorithm | Main Idea |
|---|---|---|
| CHOSEN | Dual-Objective Sector Scoring with Weighted Prefix Draw | Combines danger score, safety weight, deception bonus, prediction penalty, and prefix-sum random draw. |
| REJECTED | Two-Phase Elimination + Weighted Draw | Filters high-danger predicted sectors first, then performs weighted random selection on survivors. |
| REJECTED | Danger-Inverted Prefix Sum | Uses inverse danger and prediction penalty, but ignores Decoy_Value. |

## Step 2 — Dataset Loading and Preview

In [2]:
# ---------------------------------------------------------------------
# 0. Install & Imports
# ---------------------------------------------------------------------

import os
import random
import time
import openpyxl
import pandas as pd
from tabulate import tabulate


In [3]:
# ─────────────────────────────────────────────────────────────────────
# 1. Dataset Loading and Preview
# ─────────────────────────────────────────────────────────────────────
EXCEL_PATH = None
search_roots = ["/content/Algo-Project", os.getcwd()]
for search_root in search_roots:
    if not os.path.exists(search_root):
        continue
    for root, dirs, files in os.walk(search_root):
        for file in files:
            if "7" in file and file.endswith(".xlsx") and not file.startswith("."):
                EXCEL_PATH = os.path.join(root, file)
                break
        if EXCEL_PATH is not None:
            break
    if EXCEL_PATH is not None:
        break

if EXCEL_PATH is None:
    raise FileNotFoundError("Part 7 dataset not found. Check your repo structure.")
print(f"Dataset found: {EXCEL_PATH}")

SHEET_NAME = "B"
wb = openpyxl.load_workbook(EXCEL_PATH)
ws = wb[SHEET_NAME]

sector_columns = [
    "Sector",
    "Patrol_Frequency",
    "Thermal_Scan_Level",
    "Drone_Coverage",
    "Predicted_By_Enemy",
    "Decoy_Value",
]

sectors = []
for row in ws.iter_rows(min_row=3, values_only=True):
    if row[1] is None or row[1] == "Sector":
        continue
    sectors.append({
        "Sector"            : row[1],
        "Patrol_Frequency"  : row[2],
        "Thermal_Scan_Level": row[3],
        "Drone_Coverage"    : row[4],
        "Predicted_By_Enemy": row[5],
        "Decoy_Value"       : row[6],
    })

sector_df = pd.DataFrame(sectors, columns=sector_columns)
print(f"Loaded {len(sector_df)} sector records from Sheet '{SHEET_NAME}'\n")
print("=" * 95)
print("  PART 7 — THE PHANTOM DICE: DATASET PREVIEW")
print("=" * 95)
print(tabulate(sector_df[sector_columns], headers="keys", tablefmt="rounded_outline", showindex=False))

Dataset found: /content/Algo-Project/Datasets/Part 7.xlsx
Loaded 8 sector records from Sheet 'B'

  PART 7 — THE PHANTOM DICE: DATASET PREVIEW
╭──────────┬────────────────────┬──────────────────────┬──────────────────┬──────────────────────┬───────────────╮
│ Sector   │   Patrol_Frequency │   Thermal_Scan_Level │   Drone_Coverage │ Predicted_By_Enemy   │   Decoy_Value │
├──────────┼────────────────────┼──────────────────────┼──────────────────┼──────────────────────┼───────────────┤
│ S1       │                  2 │                    2 │                2 │ Yes                  │             4 │
│ S2       │                  5 │                    4 │                5 │ No                   │             8 │
│ S3       │                  7 │                    6 │                8 │ Yes                  │             3 │
│ S4       │                  4 │                    3 │                4 │ No                   │             7 │
│ S5       │                  6 │                   

## Step 3 — Define Helper Functions

In [4]:
def _as_sector_frame(sector_data):
    if isinstance(sector_data, pd.DataFrame):
        frame = sector_data.copy()
    else:
        frame = pd.DataFrame(sector_data)
    required = [
        "Sector", "Patrol_Frequency", "Thermal_Scan_Level",
        "Drone_Coverage", "Predicted_By_Enemy", "Decoy_Value"
    ]
    missing = [col for col in required if col not in frame.columns]
    if missing:
        raise ValueError(f"Missing required Part 7 columns: {missing}")
    for col in ["Patrol_Frequency", "Thermal_Scan_Level", "Drone_Coverage", "Decoy_Value"]:
        frame[col] = pd.to_numeric(frame[col])
    return frame[required].reset_index(drop=True)

def _draw_from_prefix(prefix_values):
    r = random.random()
    selected_idx = len(prefix_values) - 1
    for idx, threshold in enumerate(prefix_values):
        if r <= threshold:
            selected_idx = idx
            break
    return r, selected_idx

## Step 4 — Algorithm 1: Dual-Objective Sector Scoring with Weighted Prefix Draw (Chosen)

In [5]:
def dual_objective_prefix_draw(sector_data):
    t0 = time.perf_counter()
    table = _as_sector_frame(sector_data)

    table["Danger_Score"] = (
        table["Patrol_Frequency"]
        + table["Thermal_Scan_Level"]
        + table["Drone_Coverage"]
    )
    max_decoy = table["Decoy_Value"].max()
    table["Safety_Weight"] = 1 / table["Danger_Score"]
    table["Deception_Bonus"] = 1 + (table["Decoy_Value" ] / max_decoy if max_decoy else 0)
    table["Composite_Score"] = table["Safety_Weight"] * table["Deception_Bonus"]
    table.loc[table["Predicted_By_Enemy"] == "Yes", "Composite_Score"] /= 2
    table["Selection_Probability"] = table["Composite_Score"] / table["Composite_Score"].sum()
    table["Prefix_Probability"] = table["Selection_Probability"].cumsum()

    r, selected_idx = _draw_from_prefix(table["Prefix_Probability"])
    selected_real_sector = table.iloc[selected_idx].to_dict()
    real_sector_name = selected_real_sector["Sector"]

    predicted_candidates = table[
        (table["Predicted_By_Enemy"] == "Yes")
        & (table["Sector"] != real_sector_name)
    ]
    if predicted_candidates.empty:
        predicted_candidates = table[table["Sector"] != real_sector_name]
    decoy_sector = predicted_candidates.sort_values(
        ["Decoy_Value", "Composite_Score"], ascending=[False, False]
    ).iloc[0].to_dict()

    elapsed = time.perf_counter() - t0
    return {
        "selected_real_sector": selected_real_sector,
        "decoy_sector": decoy_sector,
        "probability_table": table,
        "random_value": r,
        "elapsed": elapsed,
    }

## Step 5 — Algorithm 2: Two-Phase Elimination + Weighted Draw (Rejected)

In [6]:
def two_phase_elimination_weighted_draw(sector_data):
    t0 = time.perf_counter()
    table = _as_sector_frame(sector_data)
    table["Danger_Score"] = (
        table["Patrol_Frequency"]
        + table["Thermal_Scan_Level"]
        + table["Drone_Coverage"]
    )
    sorted_dangers = sorted(table["Danger_Score"].tolist())
    middle = len(sorted_dangers) // 2
    if len(sorted_dangers) % 2 == 0:
        median_danger = (sorted_dangers[middle - 1] + sorted_dangers[middle]) / 2
    else:
        median_danger = sorted_dangers[middle]
    table["Eliminated"] = (
        (table["Danger_Score"] > median_danger)
        & (table["Predicted_By_Enemy"] == "Yes")
    )
    table["Phase_Result"] = table["Eliminated"].map({True: "Eliminated", False: "Survived"})

    survivors = table[~table["Eliminated"]].copy()
    if survivors.empty:
        survivors = table.copy()
        survivors["Phase_Result"] = "Fallback survivor"

    survivors["Weight"] = 1 / survivors["Danger_Score"]
    survivors["Selection_Probability"] = survivors["Weight"] / survivors["Weight"].sum()
    survivors["Prefix_Probability"] = survivors["Selection_Probability"].cumsum()

    r, selected_position = _draw_from_prefix(survivors["Prefix_Probability"])
    selected_sector = survivors.iloc[selected_position].to_dict()
    elapsed = time.perf_counter() - t0
    return {
        "selected_sector": selected_sector,
        "median_danger": median_danger,
        "elimination_table": table,
        "survivor_table": survivors,
        "random_value": r,
        "elapsed": elapsed,
    }

## Step 6 — Algorithm 3: Danger-Inverted Prefix Sum (Rejected)

In [7]:
def danger_inverted_prefix_sum(sector_data):
    t0 = time.perf_counter()
    table = _as_sector_frame(sector_data)
    table["Danger_Score"] = (
        table["Patrol_Frequency"]
        + table["Thermal_Scan_Level"]
        + table["Drone_Coverage"]
    )
    table["Prediction_Penalty"] = table["Predicted_By_Enemy"].map({"Yes": 0.5, "No": 1.0}).fillna(1.0)
    table["Weight"] = (1 / table["Danger_Score"]) * table["Prediction_Penalty"]
    table["Selection_Probability"] = table["Weight"] / table["Weight"].sum()
    table["Prefix_Probability"] = table["Selection_Probability"].cumsum()

    r, selected_idx = _draw_from_prefix(table["Prefix_Probability"])
    selected_sector = table.iloc[selected_idx].to_dict()
    elapsed = time.perf_counter() - t0
    return {
        "selected_sector": selected_sector,
        "probability_table": table,
        "random_value": r,
        "elapsed": elapsed,
    }

## Step 7 — Run All Candidate Algorithms

In [8]:
# ─────────────────────────────────────────────────────────────────────
# 3. Run All Three Candidate Algorithms
# ─────────────────────────────────────────────────────────────────────
print("Running all three Part 7 algorithms...\n")
dual_result = dual_objective_prefix_draw(sector_df)
two_phase_result = two_phase_elimination_weighted_draw(sector_df)
danger_inverted_result = danger_inverted_prefix_sum(sector_df)

real_sector = dual_result["selected_real_sector"]
decoy_sector = dual_result["decoy_sector"]
two_phase_sector = two_phase_result["selected_sector"]
danger_inverted_sector = danger_inverted_result["selected_sector"]

Running all three Part 7 algorithms...



## Step 8 — Chosen Algorithm Output and Probability Table

In [9]:
# ─────────────────────────────────────────────────────────────────────
# 4. Chosen Algorithm Output: Dual-Objective Sector Scoring
# ─────────────────────────────────────────────────────────────────────
chosen_table = dual_result["probability_table"]
chosen_display_columns = [
    "Sector",
    "Patrol_Frequency",
    "Thermal_Scan_Level",
    "Drone_Coverage",
    "Danger_Score",
    "Predicted_By_Enemy",
    "Decoy_Value",
    "Safety_Weight",
    "Deception_Bonus",
    "Composite_Score",
    "Selection_Probability",
    "Prefix_Probability",
]

print("=" * 120)
print("  CHOSEN ALGORITHM: DUAL-OBJECTIVE SECTOR SCORING WITH WEIGHTED PREFIX DRAW")
print("=" * 120)
print("danger_score = Patrol_Frequency + Thermal_Scan_Level + Drone_Coverage")
print("safety_weight = 1 / danger_score")
print("deception_bonus = 1 + (Decoy_Value / max_Decoy_Value)")
print("score = safety_weight × deception_bonus")
print("if Predicted_By_Enemy == Yes: score = score / 2")
print("probability = score / total_score\n")
print(tabulate(
    chosen_table[chosen_display_columns],
    headers="keys",
    tablefmt="rounded_outline",
    showindex=False,
    floatfmt=".4f",
))
print("\nNote: Prefix_Probability of last sector = 1.0000 confirms all probabilities sum correctly to 1.")

  CHOSEN ALGORITHM: DUAL-OBJECTIVE SECTOR SCORING WITH WEIGHTED PREFIX DRAW
danger_score = Patrol_Frequency + Thermal_Scan_Level + Drone_Coverage
safety_weight = 1 / danger_score
deception_bonus = 1 + (Decoy_Value / max_Decoy_Value)
score = safety_weight × deception_bonus
if Predicted_By_Enemy == Yes: score = score / 2
probability = score / total_score

╭──────────┬────────────────────┬──────────────────────┬──────────────────┬────────────────┬──────────────────────┬───────────────┬─────────────────┬───────────────────┬───────────────────┬─────────────────────────┬──────────────────────╮
│ Sector   │   Patrol_Frequency │   Thermal_Scan_Level │   Drone_Coverage │   Danger_Score │ Predicted_By_Enemy   │   Decoy_Value │   Safety_Weight │   Deception_Bonus │   Composite_Score │   Selection_Probability │   Prefix_Probability │
├──────────┼────────────────────┼──────────────────────┼──────────────────┼────────────────┼──────────────────────┼───────────────┼─────────────────┼─────────────────

In [10]:
# ---------------------------------------------------------------------
# 5. Chosen Algorithm Result: Real Sector and Decoy Sector
# ---------------------------------------------------------------------
print("=" * 95)
print("  SELECTED REAL SECTOR & DECOY SECTOR")
print("=" * 95)
print(f"Random value r                 : {dual_result['random_value']:.6f}")
print(f"Selected real sector           : {real_sector['Sector']}")
print(f"Real sector danger score       : {real_sector['Danger_Score']}")
print(f"Real sector Decoy_Value        : {real_sector['Decoy_Value']}")
print(f"Real sector predicted status   : {real_sector['Predicted_By_Enemy']}")
print(f"Decoy sector                   : {decoy_sector['Sector']}")
print(f"Decoy sector danger score      : {decoy_sector['Danger_Score']}")
print(f"Decoy sector Decoy_Value       : {decoy_sector['Decoy_Value']}")
print(f"Decoy sector predicted status  : {decoy_sector['Predicted_By_Enemy']}")
print(f"Time taken                     : {dual_result['elapsed'] * 1000:.4f} ms")
print("Note: Because this algorithm is non-deterministic, the selected real sector may differ each run.")


  SELECTED REAL SECTOR & DECOY SECTOR
Random value r                 : 0.947436
Selected real sector           : S8
Real sector danger score       : 14
Real sector Decoy_Value        : 7
Real sector predicted status   : No
Decoy sector                   : S1
Decoy sector danger score      : 6
Decoy sector Decoy_Value       : 4
Decoy sector predicted status  : Yes
Time taken                     : 21.5238 ms
Note: Because this algorithm is non-deterministic, the selected real sector may differ each run.


## Step 9 — Rejected Algorithm Outputs

### Step 9.1 — Two-Phase Elimination Output

In [11]:
# ─────────────────────────────────────────────────────────────────────
# 5. Rejected Algorithm Output: Two-Phase Elimination + Weighted Draw
# ─────────────────────────────────────────────────────────────────────
elimination_table = two_phase_result["elimination_table"]
survivor_table = two_phase_result["survivor_table"]
eliminated = elimination_table[elimination_table["Eliminated"]]

print("=" * 105)
print("  REJECTED ALGORITHM: TWO-PHASE ELIMINATION + WEIGHTED DRAW")
print("=" * 105)
print(f"Median danger score            : {two_phase_result['median_danger']:.4f}")
print(f"Random value r                 : {two_phase_result['random_value']:.6f}")
print(f"Selected sector from survivors : {two_phase_sector['Sector']}")
print(f"Time taken                     : {two_phase_result['elapsed'] * 1000:.4f} ms\n")

print("Survived sectors:")
print(tabulate(
    survivor_table[["Sector", "Danger_Score", "Predicted_By_Enemy", "Weight", "Selection_Probability", "Prefix_Probability"]],
    headers="keys",
    tablefmt="rounded_outline",
    showindex=False,
    floatfmt=".4f",
))

print("\nEliminated sectors:")
if eliminated.empty:
    print("No sectors eliminated.")
else:
    print(tabulate(
        eliminated[["Sector", "Danger_Score", "Predicted_By_Enemy", "Decoy_Value", "Phase_Result"]],
        headers="keys",
        tablefmt="rounded_outline",
        showindex=False,
        floatfmt=".4f",
    ))

  REJECTED ALGORITHM: TWO-PHASE ELIMINATION + WEIGHTED DRAW
Median danger score            : 14.0000
Random value r                 : 0.632094
Selected sector from survivors : S5
Time taken                     : 14.1403 ms

Survived sectors:
╭──────────┬────────────────┬──────────────────────┬──────────┬─────────────────────────┬──────────────────────╮
│ Sector   │   Danger_Score │ Predicted_By_Enemy   │   Weight │   Selection_Probability │   Prefix_Probability │
├──────────┼────────────────┼──────────────────────┼──────────┼─────────────────────────┼──────────────────────┤
│ S1       │              6 │ Yes                  │   0.1667 │                  0.3029 │               0.3029 │
│ S2       │             14 │ No                   │   0.0714 │                  0.1298 │               0.4328 │
│ S4       │             11 │ No                   │   0.0909 │                  0.1652 │               0.5980 │
│ S5       │             17 │ No                   │   0.0588 │                 

### Step 9.2 — Danger-Inverted Prefix Sum Output

In [12]:
# ─────────────────────────────────────────────────────────────────────
# 6. Rejected Algorithm Output: Danger-Inverted Prefix Sum
# ─────────────────────────────────────────────────────────────────────
danger_table = danger_inverted_result["probability_table"]
print("=" * 105)
print("  REJECTED ALGORITHM: DANGER-INVERTED PREFIX SUM")
print("=" * 105)
print(f"Random value r                 : {danger_inverted_result['random_value']:.6f}")
print(f"Selected sector                : {danger_inverted_sector['Sector']}")
print(f"Time taken                     : {danger_inverted_result['elapsed'] * 1000:.4f} ms\n")
print(tabulate(
    danger_table[["Sector", "Danger_Score", "Weight", "Prediction_Penalty", "Selection_Probability", "Prefix_Probability"]],
    headers="keys",
    tablefmt="rounded_outline",
    showindex=False,
    floatfmt=".4f",
))

  REJECTED ALGORITHM: DANGER-INVERTED PREFIX SUM
Random value r                 : 0.983624
Selected sector                : S8
Time taken                     : 4.9614 ms

╭──────────┬────────────────┬──────────┬──────────────────────┬─────────────────────────┬──────────────────────╮
│ Sector   │   Danger_Score │   Weight │   Prediction_Penalty │   Selection_Probability │   Prefix_Probability │
├──────────┼────────────────┼──────────┼──────────────────────┼─────────────────────────┼──────────────────────┤
│ S1       │              6 │   0.0833 │               0.5000 │                  0.1623 │               0.1623 │
│ S2       │             14 │   0.0714 │               1.0000 │                  0.1391 │               0.3015 │
│ S3       │             21 │   0.0238 │               0.5000 │                  0.0464 │               0.3478 │
│ S4       │             11 │   0.0909 │               1.0000 │                  0.1771 │               0.5249 │
│ S5       │             17 │   0.0588

## Step 10 — Time and Space Complexity Analysis

**Dual-Objective Sector Scoring:**

`T_total = O(n) + O(n) + O(n) + O(n) + O(n) + O(n) = O(n)`

`Space = O(n)`

**Two-Phase Elimination + Weighted Draw:**

`T_total = O(n) + O(n log n) + O(n) + O(n) = O(n log n)`

`Space = O(n)`

**Danger-Inverted Prefix Sum:**

`T_total = O(n) + O(n) + O(n) = O(n)`

`Space = O(n)`

## Step 11 — Algorithm Comparison Summary

In [13]:
# ─────────────────────────────────────────────────────────────────────
# 7. Algorithm Comparison Summary
# ─────────────────────────────────────────────────────────────────────
comparison_rows = [
    [
        "Dual-Objective Sector Scoring with Weighted Prefix Draw",
        "CHOSEN",
        "O(n)",
        "O(n)",
        real_sector["Sector"],
        f"{dual_result['elapsed'] * 1000:.4f}",
        "Best balance of safety, deception, and randomness",
    ],
    [
        "Two-Phase Elimination + Weighted Draw",
        "REJECTED",
        "O(n log n)",
        "O(n)",
        two_phase_sector["Sector"],
        f"{two_phase_result['elapsed'] * 1000:.4f}",
        "Sorting cost + deterministic filtering",
    ],
    [
        "Danger-Inverted Prefix Sum",
        "REJECTED",
        "O(n)",
        "O(n)",
        danger_inverted_sector["Sector"],
        f"{danger_inverted_result['elapsed'] * 1000:.4f}",
        "Ignores Decoy_Value",
    ],
]

print("=" * 125)
print("  ALGORITHM COMPARISON")
print("=" * 125)
print(tabulate(
    comparison_rows,
    headers=["Algorithm", "Status", "Time Complexity", "Space Complexity", "Selected Sector", "Time Taken (ms)", "Main Weakness / Reason"],
    tablefmt="rounded_outline",
))

  ALGORITHM COMPARISON
╭─────────────────────────────────────────────────────────┬──────────┬───────────────────┬────────────────────┬───────────────────┬───────────────────┬───────────────────────────────────────────────────╮
│ Algorithm                                               │ Status   │ Time Complexity   │ Space Complexity   │ Selected Sector   │   Time Taken (ms) │ Main Weakness / Reason                            │
├─────────────────────────────────────────────────────────┼──────────┼───────────────────┼────────────────────┼───────────────────┼───────────────────┼───────────────────────────────────────────────────┤
│ Dual-Objective Sector Scoring with Weighted Prefix Draw │ CHOSEN   │ O(n)              │ O(n)               │ S8                │           21.5238 │ Best balance of safety, deception, and randomness │
│ Two-Phase Elimination + Weighted Draw                   │ REJECTED │ O(n log n)        │ O(n)               │ S5                │           14.1403 │ Sorting c

## Step 12 — Rejection Analysis and Final Decision

**Two-Phase Elimination + Weighted Draw** is rejected because sorting makes it `O(n log n)`, deterministic elimination reduces unpredictability, and `Decoy_Value` is ignored.

**Danger-Inverted Prefix Sum** is rejected because it is `O(n)` and randomised, but it ignores `Decoy_Value`.

**Dual-Objective Sector Scoring with Weighted Prefix Draw** is chosen because it is `O(n)`, non-deterministic, and uses all five dataset columns.